- (base) root@esa-adb-gpu:~/Master_Thesis# conda create -n flam_env python=3.10 -y
- conda activate flam_env
- git clone https://github.com/owkin/FLamby.git
- cd FLamby
- make enable=fed_tcga_brca install # This installs FLamby plus only the dependencies for Fed-TCGA-BRCA.

In [4]:
import sys
print(sys.executable)
print(sys.path)

/Users/nataliamorenoblasco/anaconda3/envs/flam_env/bin/python
['/Users/nataliamorenoblasco/Desktop/Master_Thesis', '/Users/nataliamorenoblasco/anaconda3/envs/flam_env/lib/python310.zip', '/Users/nataliamorenoblasco/anaconda3/envs/flam_env/lib/python3.10', '/Users/nataliamorenoblasco/anaconda3/envs/flam_env/lib/python3.10/lib-dynload', '', '/Users/nataliamorenoblasco/anaconda3/envs/flam_env/lib/python3.10/site-packages']


In [2]:
import sys
print(sys.executable)

/root/miniconda3/envs/flamby/bin/python


In [1]:
import lifelines
print("Lifelines OK ✅")

Lifelines OK ✅


In [1]:
!which python

/root/miniconda3/envs/flamby/bin/python


# Start here:

In [1]:
from flamby.datasets.fed_tcga_brca import FedTcgaBrca

## 1. Column name extraction:

##### FLamby ready (without col names)

In [6]:
from flamby.datasets.fed_tcga_brca import FedTcgaBrca

# There are 6 centers according to the README
centers = list(range(6))

total_train = 0
total_test = 0
num_features = None

for center in centers:
    # Load train and test sets for each center
    train_ds = FedTcgaBrca(center=center, train=True)
    test_ds = FedTcgaBrca(center=center, train=False)

    total_train += len(train_ds)
    total_test += len(test_ds)

    # Get number of features from one sample
    if num_features is None and len(train_ds) > 0:
        X, y = train_ds[0]
        num_features = X.numel()

print(f"Number of centers: {len(centers)}")
print(f"Train samples total: {total_train}")
print(f"Test samples total: {total_test}")
print(f"Total samples (train + test): {total_train + total_test}")
print(f"Number of features per sample: {num_features}") # without including E and T so in total 41

Number of centers: 6
Train samples total: 866
Test samples total: 222
Total samples (train + test): 1088
Number of features per sample: 39


/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


##### brca.csv (col names)

In [7]:
import pandas as pd
import os
import flamby.datasets.fed_tcga_brca as brca_module

# 1. Automatically locate the CSV inside the FLamby package
base_path = os.path.dirname(brca_module.__file__)
csv_path = os.path.join(base_path, "brca.csv")

print(f"→ Loading: {csv_path}")

# 2. Load the CSV
df_brca = pd.read_csv(csv_path)

# 3. Display shape and column info
n_rows, n_cols = df_brca.shape
print(f"\nNumber of data points (rows): {n_rows}")
print(f"Number of columns (including targets): {n_cols}")

# 4. Inspect first few columns to see feature names
print("\nFirst 10 column names:")
print(df_brca.columns[:10].tolist())

# 5. If the file includes 'event' and 'time' at the end, subtract 2 to get #features
feature_cols = [c for c in df_brca.columns if c not in ["event", "time"]]
print(f"\nNumber of features (excluding 'event' and 'time'): {len(feature_cols)}")


→ Loading: /Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/brca.csv

Number of data points (rows): 1096
Number of columns (including targets): 42

First 10 column names:
['pid', 'age_at_index', 'ethnicity_not hispanic or latino', 'ethnicity_not reported', 'race_asian', 'race_black or african american', 'race_not reported', 'race_white', 'ajcc_pathologic_m_MX', 'ajcc_pathologic_n_N0 (i-)']

Number of features (excluding 'event' and 'time'): 42


In [9]:
import pandas as pd
import os
import flamby.datasets.fed_tcga_brca as brca_module

# 1. Locate brca.csv automatically
base_path = os.path.dirname(brca_module.__file__)
csv_path = os.path.join(base_path, "brca.csv")

# 2. Load the CSV
df_brca = pd.read_csv(csv_path)

# 3. Count missing values per column
nan_counts = df_brca.isna().sum()

# 4. Filter to show only columns that have at least one NaN
nan_columns = nan_counts[nan_counts > 0]

print(f"→ Total rows: {len(df_brca)}")
print(f"→ Columns with missing values: {len(nan_columns)}\n")
print(nan_columns)

# (Optional) summarize total NaNs
print(f"\nTotal missing values in dataset: {df_brca.isna().sum().sum()}")

→ Total rows: 1096
→ Columns with missing values: 0

Series([], dtype: int64)

Total missing values in dataset: 0


##### For FLamby ready use (the one without col names:)
- Number of centers: 6
- Total samples (train + test): 1088
- Number of features per sample: 39 (without E and T, so in total 41)

##### brca.csv --> preprocessed data (includes col names:)
- Number of data points (rows): 1096 --> 8 patients difference
- Number of columns (including targets): 42 (includes PID which is not needed, so ok same number of features in both)


In [14]:
import pandas as pd
import os
import flamby.datasets.fed_tcga_brca as brca_module

# Locate the dataset folder
base_path = os.path.dirname(brca_module.__file__)
split_path = os.path.join(base_path, "dataset_creation_scripts", "train_test_split.csv")

# Load the split file
df_split = pd.read_csv(split_path)

# Count unique pids
n_total_rows = len(df_split)
n_unique_pids = df_split["pid"].nunique()

print(f"Total rows in train_test_split.csv: {n_total_rows}")
print(f"Unique patient IDs (pids): {n_unique_pids}")

# Optional: check for duplicates
duplicates = df_split["pid"].duplicated().sum()
print(f"Duplicate patient IDs: {duplicates}")


Total rows in train_test_split.csv: 1088
Unique patient IDs (pids): 1088
Duplicate patient IDs: 0


The reason why in brca.csv there are 1096 patients and in FedTcgaBrca dataset are 1088 patients is bc the filtering logic in dataset.py. In this script, the class FedTcgaBrca loads all patient records from brca.csv but then restricts them to those whose identifiers (pid) appear in the predefined file train_test_split.csv. This split file lists exactly 1088 unique patients used for the static train/test partitions. Since 8 patient IDs from brca.csv are not included in train_test_split.csv, they are automatically excluded by the line
self.data = df2[df2["pid"].isin(pid_list)]. Thus, the 8-patient difference is not due to missing data or unmapped centers, but simply because those patients were omitted from the official split used to build the FLamby benchmark dataset.

However, the features names are still in the same order and are the same, except that in the FedTcgaBrca the PID col is omitted.

Given this, FedTcgaBrca will be used but with the features names for explainability reasons.

### Meaning of variables:
- age_at_index: The patient's age (in years) on the reference or anchor date used during date obfuscation. (integer or null)
- ethnicity_not hispanic or latino: An individual's self-described social and cultural grouping, specifically whether an individual describes themselves as Hispanic or Latino. The provided values are based on the categories defined by the U.S. Office of Management and Business and used by the U.S. Census Bureau. (binary)
    - ethnicity_not reported: binary
- race_asian: An arbitrary classification of a taxonomic group that is a division of a species. It usually arises as a consequence of geographical isolation within a species and is characterized by shared heredity, physical attributes and behavior, and in the case of humans, by common history, nationality, or geographic distribution. The provided values are based on the categories defined by the U.S. Office of Management and Business and used by the U.S. Census Bureau. (binary)
    - race_black or african american: binary
    - race_not reported: binary
    - race_white: binary
- ajcc_pathologic_m_MX: The codes that represent the stage of cancer based on the nodes present (N stage) according to criteria based on multiple editions of the AJCC's Cancer Staging Manual. (binary)
    - ajcc_pathologic_n_N0 (i-): binary
    - ajcc_pathologic_n_N1: binary
    - ajcc_pathologic_n_N1a: binary
    - ajcc_pathologic_n_N2: binary
    - ajcc_pathologic_n_N2a: binary
- ajcc_pathologic_stage_Stage IA: The extent of a cancer, especially whether the disease has spread from the original site to other parts of the body based on AJCC staging criteria. (binary)
    - ajcc_pathologic_stage_Stage IIA: binary
    - ajcc_pathologic_stage_Stage IIB: binary
    - ajcc_pathologic_stage_Stage IIIA: binary
    - ajcc_pathologic_stage_Stage IIIC: binary
- ajcc_pathologic_t_T1c: Code of pathological T (primary tumor) to define the size or contiguous extension of the primary tumor (T), using staging criteria from the American Joint Committee on Cancer (AJCC).
    - ajcc_pathologic_t_T2:
    - ajcc_pathologic_t_T3:
- ajcc_staging_system_edition_5th: The text term used to describe the version or edition of the American Joint Committee on Cancer Staging Handbooks, a publication by the group formed for the purpose of developing a system of staging for cancer that is acceptable to the American medical profession and is compatible with other accepted classifications.
    - ajcc_staging_system_edition_6th:
    - ajcc_staging_system_edition_7th:
- icd_10_code_C50.9: Alphanumeric value used to describe the disease code from the tenth version of the International Classification of Disease (ICD-10).
    - morphology_8500/3:
    - morphology_8520/3:
- "primary_diagnosis_Infiltrating duct carcinoma, NOS":Text term used to describe the patient's histologic diagnosis, as described by the World Health Organization's (WHO) International Classification of Diseases for Oncology (ICD-O).
    - "primary_diagnosis_Lobular carcinoma, NOS":
- prior_malignancy_yes: The yes/no/unknown indicator used to describe the patient's history of prior cancer diagnosis.
- synchronous_malignancy_Not Reported: A yes/no/unknown indicator used to describe whether the patient had an additional malignant diagnosis at the same time the tumor used for sequencing was diagnosed. If both tumors were sequenced, both tumors would have synchronous malignancies.
- treatment_or_therapy_not reported: A yes/no/unknown/not applicable indicator related to the administration of therapeutic agents received.
    - treatment_or_therapy_yes
- tumor_stage_stage i: The extent of a cancer in the body. Staging is usually based on the size of the tumor, whether lymph nodes contain cancer, and whether the cancer has spread from the original site to other parts of the body. The accepted values for tumor_stage depend on the tumor site, type, and accepted staging system. These items should accompany the tumor_stage value as associated metadata.
    - tumor_stage_stage ia
    - tumor_stage_stage iia
    - tumor_stage_stage iib
    - tumor_stage_stage iiia
    - tumor_stage_stage iiic
- E (event): whether the patient died (1) or was censored (0) 
    - if 0 = the patient was still alive at last follow-up (so their survival time is “censored”).
    - if 1 = event of interest (death) happened.
- T (time): time that has passed until the event  (in days). --> paper: https://arxiv.org/pdf/2006.08997
    - t represents “the time during which we have information about the patient.”
        - If the event happened → it’s the time to event.
        - If not → it’s the time to censoring (end of observation).
    

In [8]:
# Example: load training data for center 0
center0_train = FedTcgaBrca(center=0, train=True, pooled=False)

print("Center 0 - Training samples:", len(center0_train))

# Inspect one sample
X, y = center0_train[0]
print("Features shape:", X.shape)
print("Label:", y)

Center 0 - Training samples: 248
Features shape: torch.Size([39])
Label: tensor([  1., 921.])


/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


### Read data from center 0: with no features

In [ ]:
import pandas as pd
import torch
from flamby.datasets.fed_tcga_brca import FedTcgaBrca

# Load training set for center 0
center0_train = FedTcgaBrca(center=0, train=True) 

"""
train=True → loads the training split of Center 0

train=False → loads the test split of Center 0

pooled=True → ignores centers and merges everything into one pooled dataset (train or test depending on the flag)
"""

# Convert to DataFrame
features_list = []
labels_list = []

for X, y in center0_train:
    features_list.append(X.numpy())
    labels_list.append(y.numpy())

df_features = pd.DataFrame(features_list)
df_labels = pd.DataFrame(labels_list, columns=["event", "time"])

# Combine features + labels
df_center0 = pd.concat([df_features, df_labels], axis=1)

# Show all rows/columns (careful, it may be big)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Rename columns to something clearer
df_center0.columns = [f"feature_{i}" for i in range(39)] + ["event", "time"]
df_center0.head()  # or df_center0 to print everything


/root/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,event,time
0,60.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,921.0
1,43.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1325.0
2,89.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1545.0
3,50.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1142.0
4,63.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2329.0


### Read data from center 0: with features (brom brca.csv)

In [3]:
import pandas as pd
import torch
from flamby.datasets.fed_tcga_brca import FedTcgaBrca
import os

# Load training set for center 0
center0_train = FedTcgaBrca(center=0, train=True)

# --- STEP 1: Get feature names from the original CSV ---
# (You can adapt this path if needed)
csv_path = os.path.join(
    os.path.dirname(__import__('flamby').__file__),
    "datasets/fed_tcga_brca/brca.csv"
)
df_full = pd.read_csv(csv_path)

# The first column is "pid", features are columns 1:40, labels are columns 40:42
feature_names = df_full.columns[1:40].tolist()
label_names = df_full.columns[40:42].tolist()  # ["E", "T"]

# --- STEP 2: Build DataFrame from dataset tensors ---
features_list = []
labels_list = []

for X, y in center0_train:
    features_list.append(X.numpy())
    labels_list.append(y.numpy())

df_features = pd.DataFrame(features_list, columns=feature_names)
df_labels = pd.DataFrame(labels_list, columns=label_names)

# --- STEP 3: Combine features + labels ---
df_center0 = pd.concat([df_features, df_labels], axis=1)

# Optional: view all columns
pd.set_option("display.max_columns", None)

print(df_center0.head())

   age_at_index  ethnicity_not hispanic or latino  ethnicity_not reported  \
0          60.0                               0.0                     1.0   
1          43.0                               1.0                     0.0   
2          89.0                               1.0                     0.0   
3          50.0                               1.0                     0.0   
4          63.0                               0.0                     0.0   

   race_asian  race_black or african american  race_not reported  race_white  \
0         0.0                             0.0                0.0         1.0   
1         0.0                             0.0                0.0         1.0   
2         0.0                             1.0                0.0         0.0   
3         0.0                             0.0                0.0         1.0   
4         0.0                             0.0                0.0         1.0   

   ajcc_pathologic_m_MX  ajcc_pathologic_n_N0 (i-)  ajcc

/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


In [15]:
import pandas as pd
import torch
from flamby.datasets.fed_tcga_brca import FedTcgaBrca

# Load training set for center 0
center1_train = FedTcgaBrca(center=1, train=True) 

# Convert to DataFrame
features_list = []
labels_list = []

for X, y in center1_train:
    features_list.append(X.numpy())
    labels_list.append(y.numpy())

df_features = pd.DataFrame(features_list)
df_labels = pd.DataFrame(labels_list, columns=["event", "time"])

# Combine features + labels
df_center1 = pd.concat([df_features, df_labels], axis=1)

# Show all rows/columns (careful, it may be big)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Rename columns to something clearer
df_center1.columns = [f"feature_{i}" for i in range(39)] + ["event", "time"]
df_center1.head()  # or df_center0 to print everything

/root/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,event,time
0,63.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,343.0
1,41.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1604.0
2,49.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2108.0
3,61.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,2097.0
4,68.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3088.0


In [9]:
from flamby.datasets.fed_tcga_brca import TcgaBrcaRaw, FedTcgaBrca

# Raw dataset
mydataset_raw = TcgaBrcaRaw()

# Pooled test dataset
mydataset_pooled = FedTcgaBrca(train=False, pooled=True)

# Center 2 train dataset
mydataset_local2= FedTcgaBrca(center=2, train=True, pooled=False)

# Computing the length of mydataset_local2
N = len(mydataset_local2)

# Accessing individual samples
X, y = mydataset_local2[N // 2]

/root/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


In [10]:
from flamby.datasets.fed_tcga_brca import TcgaBrcaRaw

raw_dataset = TcgaBrcaRaw()
print("Raw samples:", len(raw_dataset))

Raw samples: 1096


In [12]:
import pandas as pd

features_list = []
labels_list = []

for X, y in raw_dataset:
    features_list.append(X.numpy())
    labels_list.append(y.numpy())

df_features = pd.DataFrame(features_list)
df_labels = pd.DataFrame(labels_list, columns=["event", "time"])

df_raw = pd.concat([df_features, df_labels], axis=1)

# Show nicely in Jupyter
df_raw.head()   # or just df_raw


/root/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,event,time
0,90.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,538.0
1,90.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,677.0
2,90.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,174.0
3,55.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2072.0
4,52.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,678.0
